In [1]:
# ------------------------------------------------------------------ Paso 0: setup
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np
import pyreadstat

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
%load_ext autoreload
%autoreload 2
print(RAIZ)


c:\dev\proyecto1-ehpm-desempleo


In [2]:

# ------------------------------------------------------------------ Paso 1: fijar rutas

RAW       = Path("data/raw")
INTERIM   = Path("data/interim")
PROCESSED = Path("data/processed")
DB_DIR    = Path("db")
for carpeta in (INTERIM, PROCESSED, DB_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)

In [4]:
# ------------------------------------------------------------------ Paso 2: ingesta
df, meta = pyreadstat.read_sav(RAW / "hogar_2025.sav")
print("Dimensiones:", df.shape)                         # (24534, 287)
print("Ejemplo de etiqueta:", meta.column_names_to_labels.get("CONDACT"))

Dimensiones: (28714, 342)
Ejemplo de etiqueta: Condicion de actividad


In [5]:
# ------------------------------------------------------------------ Paso 2: inventario
def construir_inventario(df, meta):
    return pd.DataFrame({
        "variable":  df.columns,
        "etiqueta":  [meta.column_names_to_labels.get(c, "") for c in df.columns],
        "dtype":     [str(df[c].dtype) for c in df.columns],
        "pct_na":    (df.isna().mean() * 100).round(1).to_numpy(),
        "n_unique":  [df[c].nunique(dropna=True) for c in df.columns],
    })

inv = construir_inventario(df, meta)
inv.to_csv(PROCESSED / "inventario_2025.csv", index=False)

In [9]:
df.head()

,HOGAR,DOMINIO,DEPTO,COR_PRE,NUM_REC,NUM_HOG,V01,V02,V02OTRO,V03,V03OTRO,V04,V04OTRO,V05,V05OTRA,V06,V07,V07OTRO,V08,V08OTRA,V09,V10,V11,V12,H01_1,H01_2,H01_3,H01_4,H01_5,H01_6,...,OCUPAOP,RAMAOS,OCUPAOS,SALMIN,NSALMIN,NUMSALMIN,ACTIVIDA,TOTHRSOP,CATEGOS,ytraop,ytraos,ytrab,yotrf,ytothg,yperhg,quintil,yhpens,yhjubi,yhAlqu,yhOtros,yhayuf,yhayup,yhReme,yheduc,yhbono,ytrabhg,yotrfhg,ymophg,Pobreza,nbi
0,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,5.0,NaN,NaN,12539.68,1.0,2.0,3.0,72.0,99.0,12000.0,NaN,12000.0,NaN,24000.0,4800.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24000.0,NaN,24000.0,2.0,0.0
1,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,5.0,NaN,NaN,12539.68,1.0,2.0,3.0,72.0,99.0,12000.0,NaN,12000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [ ]:

# ------------------------------------------------------------------ Paso 3: selección
#Definimos un diccionario nombre_estándar → variable del archivo. Trabajar con 14 variables elegidas es mejor
# que con 287. La aserción falla temprano si algún nombre no existe. base es la tabla con la que seguimos.

COLS = {
    "edad":      "EDAD",      # años cumplidos (numérica)
    "sexo":      "SEXO",      # 1 Hombre, 2 Mujer
    "area":      "UR",        # 1 Urbana, 2 Rural
    "dominio":   "DOMINIO",   # 5 zonas de estimación
    "educacion": "ED05",      # nivel educativo más alto alcanzado
    "cond_act":  "CONDACT",   # 1 Ocupado, 2 Desocupado, 3 Fuera de la FT
    "tipo_inac": "TIPINAC",   # 2 = Desalentado
    "subocup":   "ACTIVIDA",  # 2 subocupado por tiempo, 3 por ingreso
    "categoria": "CATEGOP",   # categoría ocupacional
    "rama":      "RAMAOP",    # rama de la ocupación principal
    "horas":     "TOTHRSOP",    # total de horas laboradas
    "nsalmin":   "NSALMIN",   # tramo de salario mínimo
    "salmin":    "SALMIN",    # monto del salario mínimo aplicado
    "factor":    "FACTOR",    # factor de expansión
}
faltan = [v for v in COLS.values() if v not in df.columns]
assert not faltan, f"No están en el archivo: {faltan}"
base = df[list(COLS.values())].rename(columns={v: k for k, v in COLS.items()})
print(base.shape)
base.head()

(28714, 14)


,edad,sexo,area,dominio,educacion,cond_act,tipo_inac,subocup,categoria,rama,horas,nsalmin,salmin,factor
0,60.0,1.0,1.0,1.0,4.0,1.0,NaN,3.0,2.0,7.0,72.0,1.0,12539.68,491.247861
1,53.0,2.0,1.0,1.0,4.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,491.247861
2,30.0,1.0,1.0,1.0,4.0,1.0,NaN,3.0,2.0,7.0,72.0,1.0,12539.68,491.247861
3,25.0,2.0,1.0,1.0,NaN,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,491.247861
4,15.0,2.0,1.0,1.0,NaN,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,491.247861


In [22]:
base["horas"].unique()          # array con los valores distintos
base["horas"].value_counts()    # cada valor con su frecuencia, de mayor a menor

horas
48.0     1253
40.0     1063
36.0      619
30.0      538
45.0      489
         ... 
100.0       1
111.0       1
86.0        1
117.0       1
74.0        1
Name: count, Length: 111, dtype: int64

In [17]:
# ------------------------------------------------------------------ Paso 4: tipado
#Pasamos a numéricas las que lo son, y a categorías con texto las de código. Así las tablas se leen solas y se evitan 
# promedios sin sentido sobre códigos. 

for c in ["edad", "horas", "salmin", "factor"]:
    base[c] = pd.to_numeric(base[c], errors="coerce")

etiquetas = {
    "sexo":     {1: "Hombre", 2: "Mujer"},
    "area":     {1: "Urbana", 2: "Rural"},
    "dominio":  {1: "Distrito Central", 2: "San Pedro Sula",
                 3: "Ciudades Medianas", 4: "Ciudades Pequeñas", 5: "Rural"},
    "educacion": {1: "Ninguno", 2: "Programa de alfabetización", 3: "Pre-básica (1-3)", 4: "Básica (1-9)", 5: "Media/Diversificado (1-4)",
                  6: "Técnico superior (1-3)", 7: "Superior no universitaria (1-4)", 8: "Superior universitaria (1-8)", 
                  9: "Especialidad (1-3)", 10: "Maestria (1-2)", 11: "Doctorado (1-4)", 99: "No sabe /  No responde"},
    "cond_act": {1: "Ocupado", 2: "Desocupado", 3: "Fuera de la fuerza"},
    "tipo_inac": {1: "Potencialmente Activos", 2: "Desalentados"},
    "subocup": {1: "Sin problemas de empleo", 2: "Subocupados por tiempo de trabajo", 3: "Subocupados por ingreso"},
    "categoria" : {1: "Empleado u obrero público", 2: "Empleado u obrero privado", 3: "Empleado doméstico", 4: "Cuenta propia", 
                    5: "Aprendiz", 6: "Trabajador familiar", 7: "Contratista dependiente", 8: "No declaró categoría"},
    "rama" : {1: "Agricultura, ganadería, silvicultura y pesca", 2: "Explotacion de minas y canteras", 3: "Manufacturera", 
              4: "Suministro de electricidad, gas, vapor y aire acondicionado", 5: "Suministro de agua evacuación de aguas residuales, gestión de desechos y descontaminación", 
              6: "Construcción", 7: "Comercio al por mayor y al por menor reparación de vehículos automotores y motocicletas", 
              8: "Transporte y almacenamiento", 9: "Actividades de alojamiento y de servicio de comidas", 10: "Información y comunicaciones", 
              11: "Actividades financieras y de seguros", 12: "Actividades inmobiliarias", 13: "Actividades profesionales, científicas y técnicas", 
              14: "Actividades de servicios administrativos y de apoyo", 15: "Administración pública y defensa planes de seguridad social de afiliación obligatoria", 
              16: "Enseñanza", 17: "Actividades de atención de la salud humana y de asistencia social", 18: "Actividades artísticas, de entretenimiento y recreativas",
              19: "Otras actividades de servicios", 20: "Actividades de los hogares como empleadores actividades no diferenciadas de los hogares como productores de bienes y ser",
              21: "Actividades de organizaciones y órganos extraterritoriales", 22: "Rama sin especificar", 23: "Busca trabajo por primera vez",
              99: "Ns/Nr"}
}
for c, mapa in etiquetas.items():
    base[c] = base[c].map(mapa).astype("category")
base.dtypes

edad          float64
sexo         category
area         category
dominio      category
educacion    category
cond_act     category
tipo_inac    category
subocup      category
categoria    category
rama         category
horas         float64
nsalmin       float64
salmin        float64
factor        float64
dtype: object